# cctv_dfd — full training run on local 10K-frame dataset

Real run that produces report-grade numbers. Uses the existing
10,905-frame dataset (Real + Fake split from the legacy preprocessing).
No FF++/Celeb-DF download needed.

**What this produces:**
- `results/checkpoints/local.pt` — DINOv2-S + MLP head trained ~20 epochs
- `results/metrics/local.json` — train/val curves + best epoch
- `results/metrics/local_eval.json` — 5 profiles × 3 enhance modes = 15-cell matrix
- `results/metrics/local_eval.md` — markdown table for the report
- A handful of attention-overlay PNGs in `reports/`

**Runtime on T4:** ~30-40 minutes total.

## 1. Pull the latest code

In [ ]:
%cd /content/repo
!git pull
%cd /content/repo/cctv_dfd

In [ ]:
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv,noheader

## 2. Build the full local dataset (all ~10,905 frames)

In [ ]:
# Idempotent: skips files already present. Safe to re-run.
!python -m cctv_dfd.cli face-extract \
  --input /content/repo/Deepfake_Detection_Project/Deepfake_Detection_Project/data \
  --output data/processed/local \
  --already-cropped

In [ ]:
!echo Real: $(ls data/processed/local/Real | wc -l)
!echo Fake: $(ls data/processed/local/Fake | wc -l)

## 3. Train the head (20 epochs, ~10-15 min on T4)

In [ ]:
# Remove old smoke-test artifacts to keep results/ clean.
!rm -f results/checkpoints/smoke.pt results/metrics/smoke*.json reports/*.png reports/*.html results/forensic.jsonl

In [ ]:
!python -m cctv_dfd.cli train \
  --config configs/dinov2s_mlp.yaml \
  --dataset local --epochs 20 --batch-size 64 --tag local

## 4. Stratified evaluation (5 profiles × 3 enhance modes)

In [ ]:
# 'aggressive' enhance mode loads Real-ESRGAN; if unavailable it falls
# back to 'forensic' (logged warning). Drop 'aggressive' here if you
# don't want to install the extra dependency.
!python -m cctv_dfd.cli eval \
  --run-tag local --dataset local \
  --profiles all --enhance-modes none forensic

In [ ]:
import json, pathlib
data = json.loads(pathlib.Path('results/metrics/local_eval.json').read_text())
rows = data['rows']
print(f"{'profile':16s} {'enhance':9s} {'acc':>6s} {'prec':>6s} {'rec':>6s} {'f1':>6s} {'auc':>6s} {'eer':>6s} {'n':>4s}")
print('-' * 72)
for r in rows:
    print(f"{r['profile']:16s} {r['enhance']:9s} "
          f"{r['acc']:6.3f} {r['precision']:6.3f} {r['recall']:6.3f} "
          f"{r['f1']:6.3f} {r['auc']:6.3f} {r['eer']:6.3f} {r['n']:4d}")

In [ ]:
# Render the markdown table directly in the notebook.
from IPython.display import Markdown, display
display(Markdown(pathlib.Path('results/metrics/local_eval.md').read_text()))

## 5. Headline figures for the report

Two graphs the report needs:
1. AUC vs degradation profile (one line per enhance mode)
2. Train/val loss curves

In [ ]:
import matplotlib.pyplot as plt
from collections import defaultdict
by_enh = defaultdict(list)
for r in rows:
    by_enh[r['enhance']].append((r['profile'], r['auc']))

fig, ax = plt.subplots(figsize=(8, 4.5))
from cctv_dfd.degradations import PROFILE_NAMES
order = list(PROFILE_NAMES)
for enh, pairs in by_enh.items():
    d = dict(pairs)
    ax.plot(order, [d[p] for p in order], marker='o', label=f'enhance={enh}')
ax.set_xlabel('degradation profile')
ax.set_ylabel('AUC-ROC')
ax.set_ylim(0.5, 1.0)
ax.set_title('AUC across CCTV degradation profiles (local dataset)')
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig('results/metrics/local_auc_by_profile.png', dpi=120)
plt.show()

In [ ]:
tr = json.loads(pathlib.Path('results/metrics/local.json').read_text())
hist = tr['history']
epochs = [h['epoch'] for h in hist]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(epochs, [h['train_loss'] for h in hist], label='train loss', marker='o')
ax.plot(epochs, [h['val_loss'] for h in hist], label='val loss', marker='s')
ax2 = ax.twinx()
ax2.plot(epochs, [h['val_auc'] for h in hist], color='tab:green', label='val AUC', marker='^')
ax.set_xlabel('epoch'); ax.set_ylabel('loss'); ax2.set_ylabel('AUC')
ax.legend(loc='upper left'); ax2.legend(loc='lower right')
ax.set_title('Training curves')
plt.tight_layout()
plt.savefig('results/metrics/local_training_curves.png', dpi=120)
plt.show()
print('best epoch:', tr['best_epoch'], 'best val_auc:', tr['best_val_auc'])

## 6. Generate attention-rollout explanations for 4 samples

Two reals + two fakes. These PNGs go in the report's explainability section.

In [ ]:
import pathlib
reals = sorted(pathlib.Path('data/processed/local/Real').glob('*.jpg'))[:2]
fakes = sorted(pathlib.Path('data/processed/local/Fake').glob('*.jpg'))[:2]
samples = reals + fakes
for s in samples:
    print('---', s)
    !python -m cctv_dfd.cli predict --run-tag local --image {s} --enhance forensic

In [ ]:
# Show the attention overlays inline.
from IPython.display import Image, display
for s in samples:
    png = pathlib.Path('reports') / f'{s.stem}_explain.png'
    if png.exists():
        print(s.name)
        display(Image(str(png)))

In [ ]:
# Verify forensic log integrity
import json
with open('results/forensic.jsonl') as f:
    for line in f:
        rec = json.loads(line)
        print(f"{rec['ts']}  {rec['prediction']:4s}  p={rec['prob_fake']:.3f}  "
              f"prof={rec['degradation_profile']:14s} enh={rec['enhance_mode']:9s} "
              f"input_sha={rec['input_sha256'][:12]}")

## 7. Download artifacts for the report

Zips up `results/` and `reports/` so you can pull them off Colab in one shot.

In [ ]:
!cd /content/repo/cctv_dfd && zip -qr /content/cctv_dfd_results.zip results/ reports/ && ls -lh /content/cctv_dfd_results.zip
# In Colab: files panel -> right-click /content/cctv_dfd_results.zip -> Download

## What's now in the report

- `local_auc_by_profile.png` → figure for §8 (Results) showing the *degradation-stratified AUC*, the headline finding
- `local_training_curves.png` → figure for §7 (Implementation) showing training dynamics
- `local_eval.md` → drop the markdown table directly into §8
- 4 `*_explain.png` overlays → figure(s) for the explainability section
- `forensic.jsonl` → cite as the chain-of-custody trace in §4.2
- `local.pt` → the trained checkpoint, ~400 KB, archive it